In [1]:
import numpy as np

In [6]:
import os

folder_path = 'E:/Semisters/brain tumer prediction/Master Dataset'

if os.path.exists(folder_path):
    print(f"Contents of '{folder_path}':")
    for item in os.listdir(folder_path):
        print(item)
else:
    print(f"Error: Folder not found at {folder_path}")

Contents of 'E:/Semisters/brain tumer prediction/Master Dataset':
Astrocitoma
Ependimoma
Germinoma
Glioblastoma
Meduloblastoma
meningioma
no_tumor
Oligodendroglioma
pituitary
Schwannoma


In [7]:
import os

base_folder_path = 'E:/Semisters/brain tumer prediction/Master Dataset'

# List of subdirectories (your 'classes')
subdirectories = [
    'no_tumor',
    'Meduloblastoma',
    'Germinoma',
    'Oligodendroglioma',
    'Astrocitoma',
    'pituitary',
    'meningioma',
    'Ependimoma',
    'Glioblastoma',
    'Schwannoma'
]

image_extensions = ('.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff', '.webp') # Common image file extensions

print(f"Counting files in subdirectories of: {base_folder_path}\n")

for sub_dir_name in subdirectories:
    sub_dir_path = os.path.join(base_folder_path, sub_dir_name)

    if os.path.exists(sub_dir_path) and os.path.isdir(sub_dir_path):
        file_count = 0
        for item in os.listdir(sub_dir_path):
            if os.path.isfile(os.path.join(sub_dir_path, item)) and item.lower().endswith(image_extensions):
                file_count += 1
        print(f"  '{sub_dir_name}': {file_count} image files")
    else:
        print(f"  Warning: Subdirectory '{sub_dir_name}' not found or is not a directory.")


Counting files in subdirectories of: E:/Semisters/brain tumer prediction/Master Dataset

  'no_tumor': 2396 image files
  'Meduloblastoma': 131 image files
  'Germinoma': 100 image files
  'Oligodendroglioma': 224 image files
  'Astrocitoma': 580 image files
  'pituitary': 2658 image files
  'meningioma': 2582 image files
  'Ependimoma': 150 image files
  'Glioblastoma': 204 image files
  'Schwannoma': 465 image files


In [ ]:
# Brain Tumor MRI Detection - Preprocessing Pipeline
# Python 3.8+

# Core Dependencies
#numpy >=1.21.0
#opencv-python >=4.5.0
#Pillow >=8.0.0

# Data Visualization
#matplotlib >=3.4.0

# Progress Bar
#tqdm >=4.62.0

# Optional: For deep learning (install separately when training)
# tensorflow>=2.8.0
# torch>=1.10.0
# torchvision>=0.11.0
# scikit-learn>=1.0.0
# pandas>=1.3.0

In [14]:
import os
import shutil

master_dir = "E:/Semisters/brain tumer prediction/Master Dataset"

# Define the rename mappings
renames = {
    'no_tumor': 'No_Tumor',
    'Meduloblastoma': 'Medulloblastoma',
    'Astrocitoma': 'Astrocytoma',
    'pituitary': 'Pituitary',
    'meningioma': 'Meningioma',
    'Ependimoma': 'Ependymoma'
}

# Perform renames
print("🔧 Renaming folders...\n")
for old_name, new_name in renames.items():
    old_path = os.path.join(master_dir, old_name)
    new_path = os.path.join(master_dir, new_name)

    if os.path.exists(old_path):
        try:
            os.rename(old_path, new_path)
            print(f"✅ Renamed: '{old_name}' → '{new_name}'")
        except Exception as e:
            print(f"❌ Error renaming '{old_name}': {str(e)}")
    else:
        print(f"⚠️  Folder '{old_name}' not found")

print("\n✅ All folders renamed!")
print("\n🚀 Now you can run preprocessing:")
print("   %run preprocess_brain_mri.py")

🔧 Renaming folders...

✅ Renamed: 'no_tumor' → 'No_Tumor'
✅ Renamed: 'Meduloblastoma' → 'Medulloblastoma'
✅ Renamed: 'Astrocitoma' → 'Astrocytoma'
✅ Renamed: 'pituitary' → 'Pituitary'
✅ Renamed: 'meningioma' → 'Meningioma'
✅ Renamed: 'Ependimoma' → 'Ependymoma'

✅ All folders renamed!

🚀 Now you can run preprocessing:
   %run preprocess_brain_mri.py


In [15]:
import os

folder_path = 'E:/Semisters/brain tumer prediction/Master Dataset'

if os.path.exists(folder_path):
    print(f"Contents of '{folder_path}':")
    for item in os.listdir(folder_path):
        print(item)
else:
    print(f"Error: Folder not found at {folder_path}")

Contents of 'E:/Semisters/brain tumer prediction/Master Dataset':
Astrocytoma
Ependymoma
Germinoma
Glioblastoma
Medulloblastoma
Meningioma
No_Tumor
Oligodendroglioma
Pituitary
Schwannoma


In [16]:
"""
Brain Tumor MRI Image Preprocessing Pipeline
Author: Your Name
Project: 10-Class Brain Tumor Detection

This script performs comprehensive preprocessing on MRI images:
1. Removes black background (skull stripping/cropping)
2. Resizes to 224x224
3. Converts to RGB
4. Normalizes pixel values
5. Saves processed images to output directory
"""

import os
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm
import shutil
import json
from datetime import datetime


class BrainMRIPreprocessor:
    """
    Preprocessor for Brain MRI images with 10 tumor classes
    """

    def __init__(self, input_dir, output_dir, target_size=(224, 224)):
        """
        Initialize the preprocessor

        Args:
            input_dir (str): Path to brain_mri_10class directory
            output_dir (str): Path to save processed images
            target_size (tuple): Target image size (width, height)
        """
        self.input_dir = Path(input_dir)
        self.output_dir = Path(output_dir)
        self.target_size = target_size

        # Define the 10 classes (LOCKED 🔒)
        self.classes = [
            'Astrocytoma',
            'Ependymoma',
            'Glioblastoma',
            'Germinoma',
            'Medulloblastoma',
            'Meningioma',
            'Oligodendroglioma',
            'Pituitary',
            'Schwannoma',
            'No_Tumor'
        ]

        self.stats = {
            'total_images': 0,
            'processed_images': 0,
            'failed_images': 0,
            'class_distribution': {},
            'skipped_images': []
        }

    def remove_black_background(self, image, threshold=10):
        """
        Remove black background from MRI image using contour detection

        Args:
            image: Input grayscale or RGB image
            threshold: Pixel intensity threshold for background detection

        Returns:
            Cropped image with black background removed
        """
        # Convert to grayscale if needed
        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        else:
            gray = image.copy()

        # Create binary mask (pixels > threshold are foreground)
        _, binary = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)

        # Find contours
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if not contours:
            # If no contours found, return original image
            return image

        # Find the largest contour (assumed to be the brain)
        largest_contour = max(contours, key=cv2.contourArea)

        # Get bounding box
        x, y, w, h = cv2.boundingRect(largest_contour)

        # Add small padding (5% of dimensions)
        padding_x = int(w * 0.05)
        padding_y = int(h * 0.05)

        x = max(0, x - padding_x)
        y = max(0, y - padding_y)
        w = min(image.shape[1] - x, w + 2 * padding_x)
        h = min(image.shape[0] - y, h + 2 * padding_y)

        # Crop the image
        cropped = image[y:y+h, x:x+w]

        return cropped

    def normalize_image(self, image):
        """
        Normalize image pixel values to [0, 1] range

        Args:
            image: Input image

        Returns:
            Normalized image (float32)
        """
        # Convert to float and normalize to [0, 1]
        normalized = image.astype(np.float32) / 255.0
        return normalized

    def preprocess_single_image(self, image_path, apply_normalization=True):
        """
        Apply complete preprocessing pipeline to a single image

        Args:
            image_path: Path to the input image
            apply_normalization: Whether to normalize pixel values

        Returns:
            Preprocessed image or None if processing fails
        """
        try:
            # Read image
            image = cv2.imread(str(image_path))

            if image is None:
                print(f"⚠️  Failed to read: {image_path}")
                return None

            # Step 1: Remove black background
            cropped = self.remove_black_background(image)

            # Step 2: Resize to target size
            resized = cv2.resize(cropped, self.target_size, interpolation=cv2.INTER_AREA)

            # Step 3: Convert to RGB (OpenCV reads as BGR)
            rgb_image = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)

            # Step 4: Normalize (optional, can be done later in training)
            if apply_normalization:
                processed = self.normalize_image(rgb_image)
                # Convert back to uint8 for saving
                processed = (processed * 255).astype(np.uint8)
            else:
                processed = rgb_image

            return processed

        except Exception as e:
            print(f"❌ Error processing {image_path}: {str(e)}")
            return None

    def process_dataset(self, save_normalized=False):
        """
        Process entire dataset

        Args:
            save_normalized: If True, saves normalized [0,1] images as float32 numpy arrays
                           If False, saves as uint8 PNG images
        """
        print("=" * 80)
        print("🧠 BRAIN TUMOR MRI PREPROCESSING PIPELINE")
        print("=" * 80)
        print(f"Input directory: {self.input_dir}")
        print(f"Output directory: {self.output_dir}")
        print(f"Target size: {self.target_size}")
        print(f"Classes: {len(self.classes)}")
        print("=" * 80)

        # Create output directory structure
        self.output_dir.mkdir(parents=True, exist_ok=True)

        for class_name in self.classes:
            class_dir = self.output_dir / class_name
            class_dir.mkdir(exist_ok=True)

        # Process each class
        for class_name in self.classes:
            print(f"\n📁 Processing class: {class_name}")

            class_input_dir = self.input_dir / class_name
            class_output_dir = self.output_dir / class_name

            if not class_input_dir.exists():
                print(f"⚠️  Warning: {class_input_dir} does not exist. Skipping...")
                continue

            # Get all image files
            image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff']
            image_files = []
            for ext in image_extensions:
                image_files.extend(list(class_input_dir.glob(f'*{ext}')))
                image_files.extend(list(class_input_dir.glob(f'*{ext.upper()}')))

            if not image_files:
                print(f"⚠️  No images found in {class_input_dir}")
                continue

            self.stats['total_images'] += len(image_files)
            class_processed = 0
            class_failed = 0

            # Process images with progress bar
            for img_path in tqdm(image_files, desc=f"  Processing {class_name}"):
                # Preprocess image
                processed_img = self.preprocess_single_image(
                    img_path,
                    apply_normalization=False  # Save as uint8 PNG
                )

                if processed_img is not None:
                    # Save processed image
                    output_path = class_output_dir / f"{img_path.stem}_processed.png"

                    # Convert RGB back to BGR for cv2.imwrite
                    bgr_image = cv2.cvtColor(processed_img, cv2.COLOR_RGB2BGR)
                    success = cv2.imwrite(str(output_path), bgr_image)

                    if success:
                        class_processed += 1
                        self.stats['processed_images'] += 1
                    else:
                        class_failed += 1
                        self.stats['failed_images'] += 1
                        self.stats['skipped_images'].append(str(img_path))
                else:
                    class_failed += 1
                    self.stats['failed_images'] += 1
                    self.stats['skipped_images'].append(str(img_path))

            self.stats['class_distribution'][class_name] = class_processed
            print(f"  ✅ Processed: {class_processed} | ❌ Failed: {class_failed}")

        # Print final statistics
        self.print_statistics()

        # Save statistics to JSON
        self.save_statistics()

    def print_statistics(self):
        """Print preprocessing statistics"""
        print("\n" + "=" * 80)
        print("📊 PREPROCESSING STATISTICS")
        print("=" * 80)
        print(f"Total images found: {self.stats['total_images']}")
        print(f"Successfully processed: {self.stats['processed_images']}")
        print(f"Failed: {self.stats['failed_images']}")
        print(f"Success rate: {self.stats['processed_images']/max(1, self.stats['total_images'])*100:.2f}%")
        print("\n📈 Class Distribution:")
        print("-" * 40)
        for class_name, count in sorted(self.stats['class_distribution'].items()):
            print(f"  {class_name:20s}: {count:5d} images")
        print("=" * 80)

    def save_statistics(self):
        """Save statistics to JSON file"""
        stats_file = self.output_dir / 'preprocessing_stats.json'
        stats_data = {
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'input_directory': str(self.input_dir),
            'output_directory': str(self.output_dir),
            'target_size': list(self.target_size),
            'statistics': self.stats
        }

        with open(stats_file, 'w') as f:
            json.dump(stats_data, f, indent=2)

        print(f"\n💾 Statistics saved to: {stats_file}")


def main():
    """
    Main execution function
    """
    # Configuration
    INPUT_DIR = "E:/Semisters/brain tumer prediction/Master Dataset"  # Your master dataset directory
    OUTPUT_DIR = "brain_mri_10class_processed"  # Output directory for processed images
    TARGET_SIZE = (224, 224)  # Standard size for CNN models (ResNet, VGG, etc.)

    print("\n🚀 Starting Brain MRI Preprocessing Pipeline\n")

    # Check if input directory exists
    if not os.path.exists(INPUT_DIR):
        print(f"❌ Error: Input directory '{INPUT_DIR}' not found!")
        print(f"Please ensure your master dataset is in the '{INPUT_DIR}' folder")
        print("\nExpected structure:")
        print("brain_mri_10class/")
        print("├── Astrocytoma/")
        print("├── Ependymoma/")
        print("├── Glioblastoma/")
        print("├── Germinoma/")
        print("├── Medulloblastoma/")
        print("├── Meningioma/")
        print("├── Oligodendroglioma/")
        print("├── Pituitary/")
        print("├── Schwannoma/")
        print("└── No_Tumor/")
        return

    # Create preprocessor
    preprocessor = BrainMRIPreprocessor(
        input_dir=INPUT_DIR,
        output_dir=OUTPUT_DIR,
        target_size=TARGET_SIZE
    )

    # Process dataset
    preprocessor.process_dataset(save_normalized=False)

    print("\n✅ Preprocessing completed!")
    print(f"📂 Processed images saved to: {OUTPUT_DIR}")
    print("\n🔄 Next step: Run train_val_test_split.py to split the data")


if __name__ == "__main__":
    main()


🚀 Starting Brain MRI Preprocessing Pipeline

🧠 BRAIN TUMOR MRI PREPROCESSING PIPELINE
Input directory: E:\Semisters\brain tumer prediction\Master Dataset
Output directory: brain_mri_10class_processed
Target size: (224, 224)
Classes: 10

📁 Processing class: Astrocytoma


  Processing Astrocytoma: 100%|██████████| 1158/1158 [00:22<00:00, 50.85it/s]


  ✅ Processed: 1158 | ❌ Failed: 0

📁 Processing class: Ependymoma


  Processing Ependymoma: 100%|██████████| 300/300 [00:07<00:00, 40.83it/s]


  ✅ Processed: 300 | ❌ Failed: 0

📁 Processing class: Glioblastoma


...

  ✅ Processed: 5316 | ❌ Failed: 0

📁 Processing class: Schwannoma


  Processing Schwannoma: 100%|██████████| 930/930 [00:20<00:00, 46.36it/s]


  ✅ Processed: 930 | ❌ Failed: 0

📁 Processing class: No_Tumor


  Processing No_Tumor: 100%|██████████| 4792/4792 [01:22<00:00, 58.19it/s] 


  ✅ Processed: 4792 | ❌ Failed: 0

📊 PREPROCESSING STATISTICS
Total images found: 18978
Successfully processed: 18978
Failed: 0
Success rate: 100.00%

📈 Class Distribution:
----------------------------------------
  Astrocytoma         :  1158 images
  Ependymoma          :   300 images
  Germinoma           :   200 images
  Glioblastoma        :   408 images
  Medulloblastoma     :   262 images
  Meningioma          :  5164 images
  No_Tumor            :  4792 images
  Oligodendroglioma   :   448 images
  Pituitary           :  5316 images
  Schwannoma          :   930 images

💾 Statistics saved to: brain_mri_10class_processed\preprocessing_stats.json

✅ Preprocessing completed!
📂 Processed images saved to: brain_mri_10class_processed

🔄 Next step: Run train_val_test_split.py to split the data
